In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib import colormaps, font_manager
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import linkage, leaves_list


DATA = Path('risk_allele_plot')

OUT_PNG = DATA / 'ad_common_augmented_main_heatmap_with_tie_direction_corrected.png'
OUT_PDF = DATA / 'ad_common_augmented_main_heatmap_with_tie_direction_corrected.pdf'

COMMON_FILE = DATA / 'common_Bellenguez_variant_votes.csv'
RARE_FILE = DATA / 'rare_Prokopenko_variant_votes.csv'
CLASS_FILE = DATA / 'gwas_wgs_locus_source_classification.csv'


# IMPORTANT DIRECTION CORRECTION
# ------------------------------
# The upstream vote CSVs were generated under the assumption that
# logFC_converted = Minor - Major. The actual convention is:
#
#     logFC_converted = Major - Minor
#
# Therefore, every non-tied association-aligned vote in those existing CSVs
# has the opposite direction. Swapping increase <-> decrease after the
# variant-level vote is mathematically equivalent to correcting every
# construct before voting; ties remain ties.
CORRECT_REVERSED_LOGFC_DIRECTION = True


GWAS_LOCI = [
    'HLA-DRB1',
    'BIN1',
    'MS4A4A',
    'ABCA7',
    'CLU',
    'SCIMP',
    'HLA-DQA1',
    'SORL1',
    'PTK2B',
    'APOC2',
]

WGS_LOCI = [
    'APOE',
    'OSTM1',
    'NELL2',
    'DBX2',
    'POU3F3',
    'IL1F10',
    'ST8SIA4',
    'SNAP25',
    'BTBD2',
    'PRKCH',
]

DISPLAY = {
    'OSTM1': 'SEC63–OSTM1',
}

ASSAYS = [
    'THP-1 macrophages',
    'Brain tissue',
]

# Six cells per row: increase, decrease, and tie for each assay.
COLS = [
    ('THP-1 macrophages', 'increase'),
    ('THP-1 macrophages', 'decrease'),
    ('THP-1 macrophages', 'tie'),
    ('Brain tissue', 'increase'),
    ('Brain tissue', 'decrease'),
    ('Brain tissue', 'tie'),
]

# Extra space between THP-1 and Brain.
XPOS = [0.0, 1.0, 2.0, 4.0, 5.0, 6.0]


# Figure typography
GENE_FONTSIZE = 16
COUNT_FONTSIZE = 21
XTICK_FONTSIZE = 18
GROUP_FONTSIZE = 17
TITLE_FONTSIZE = 20
PANEL_FONTSIZE = 24
LEGEND_FONTSIZE = 12

CELL_W = 0.98
CELL_H = 0.96


def set_style():
    preferred_fonts = [
        'Arial',
        'Helvetica',
        'Liberation Sans',
        'DejaVu Sans',
    ]

    available = {
        font.name
        for font in font_manager.fontManager.ttflist
    }

    chosen = next(
        (
            font
            for font in preferred_fonts
            if font in available
        ),
        'DejaVu Sans',
    )

    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': [chosen] + preferred_fonts,
        'font.size': 15,
        'axes.titlesize': TITLE_FONTSIZE,
        'axes.labelsize': 16,
        'xtick.labelsize': XTICK_FONTSIZE,
        'ytick.labelsize': GENE_FONTSIZE,
        'axes.linewidth': 1.0,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'svg.fonttype': 'none',
        'savefig.dpi': 600,
    })

    print('Using font:', chosen)


def prepare_vote_table(df: pd.DataFrame, source_class: str) -> pd.DataFrame:
    """
    Convert each variant-level vote into exactly one of three categories.

    This deliberately ignores the old count_increase/count_decrease fields for
    tied variants, because those files may encode a tie as one count in both
    directions. A tied vote is instead counted only in count_tie.
    """
    required = ['assay', 'locus', 'vote']
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f'Missing required columns for {source_class}: {missing}. '
            'The input vote file must contain assay, locus, and vote.'
        )

    out = df[required].copy()
    out['vote'] = out['vote'].astype(str).str.strip().str.lower()

    valid_votes = {'increase', 'decrease', 'tie'}
    invalid = sorted(set(out['vote'].dropna()) - valid_votes)
    if invalid:
        raise ValueError(
            f'Unexpected vote labels in {source_class} file: {invalid}'
        )

    # Correct the global sign inversion in the precomputed vote files.
    # With logFC_converted = Major - Minor:
    #   - positive logFC means Major has higher reporter transcription;
    #   - negative logFC means Minor has higher reporter transcription.
    # The original vote files used the opposite convention, so all
    # non-tied risk-/W-aligned votes must be inverted.
    if CORRECT_REVERSED_LOGFC_DIRECTION:
        out['vote'] = out['vote'].replace({
            'increase': 'decrease',
            'decrease': 'increase',
            'tie': 'tie',
        })

    out['count_increase'] = (out['vote'] == 'increase').astype(int)
    out['count_decrease'] = (out['vote'] == 'decrease').astype(int)
    out['count_tie'] = (out['vote'] == 'tie').astype(int)
    out['source_class'] = source_class
    return out


def load_counts():
    common_raw = pd.read_csv(COMMON_FILE)
    rare_raw = pd.read_csv(RARE_FILE)
    cls = pd.read_csv(CLASS_FILE)

    common = prepare_vote_table(common_raw, 'GWAS')
    rare = prepare_vote_table(rare_raw, 'WGS')

    votes = pd.concat(
        [common, rare],
        ignore_index=True,
    )

    # Enforce the classification file. Mixed-source loci remain assigned
    # according to that file, which should already implement GWAS priority.
    votes = votes.merge(
        cls[['locus', 'source_class']],
        on='locus',
        how='left',
        suffixes=('', '_cls'),
    )

    votes['source_class'] = (
        votes['source_class_cls']
        .fillna(votes['source_class'])
    )

    votes = votes.drop(
        columns=['source_class_cls']
    )

    counts = (
        votes
        .groupby(
            [
                'locus',
                'source_class',
                'assay',
            ],
            as_index=False,
        )[
            [
                'count_increase',
                'count_decrease',
                'count_tie',
            ]
        ]
        .sum()
    )

    # Complete the grid for all selected loci.
    rows = []

    for locus in GWAS_LOCI:
        for assay in ASSAYS:
            rows.append(
                (locus, 'GWAS', assay)
            )

    for locus in WGS_LOCI:
        for assay in ASSAYS:
            rows.append(
                (locus, 'WGS', assay)
            )

    base = pd.DataFrame(
        rows,
        columns=[
            'locus',
            'source_class',
            'assay',
        ],
    )

    counts = base.merge(
        counts,
        on=[
            'locus',
            'source_class',
            'assay',
        ],
        how='left',
    )

    count_columns = [
        'count_increase',
        'count_decrease',
        'count_tie',
    ]

    counts[count_columns] = (
        counts[count_columns]
        .fillna(0)
        .astype(int)
    )

    # The three categories are mutually exclusive, so the denominator is the
    # number of unique voted variants in that locus and assay.
    counts['total'] = counts[count_columns].sum(axis=1)

    for direction in ['increase', 'decrease', 'tie']:
        counts[f'{direction}_fraction'] = np.where(
            counts['total'] > 0,
            counts[f'count_{direction}'] / counts['total'],
            0.0,
        )

    return counts


def matrix_for_loci(counts, loci):
    lookup = counts.set_index(
        ['locus', 'assay']
    )

    values = np.zeros(
        (len(loci), len(COLS)),
        dtype=int,
    )

    fracs = np.zeros(
        (len(loci), len(COLS)),
        dtype=float,
    )

    for i, locus in enumerate(loci):
        for j, (assay, direction) in enumerate(COLS):
            row = lookup.loc[(locus, assay)]

            values[i, j] = int(
                row[f'count_{direction}']
            )

            fracs[i, j] = float(
                row[f'{direction}_fraction']
            )

    return values, fracs


def cluster_loci(counts, loci):
    if len(loci) <= 2:
        return loci

    values, fracs = matrix_for_loci(
        counts,
        loci,
    )

    totals = values.sum(
        axis=1,
        keepdims=True,
    )

    totals = np.where(
        totals == 0,
        1,
        totals,
    )

    normalized_values = values / totals

    features = np.concatenate(
        [
            fracs,
            normalized_values,
        ],
        axis=1,
    )

    try:
        linkage_matrix = linkage(
            features,
            method='average',
            metric='euclidean',
            optimal_ordering=True,
        )

        order = leaves_list(
            linkage_matrix
        )

        return [
            loci[i]
            for i in order
        ]

    except Exception as error:
        print('Clustering failed:', error)
        return loci


def get_color(direction, frac):
    if frac <= 0:
        return (1, 1, 1, 1)

    cmap = {
        'increase': colormaps['Reds'],
        'decrease': colormaps['Blues'],
        'tie': colormaps['Greys'],
    }[direction]

    return cmap(
        0.18 + 0.82 * frac
    )


def get_number_fontsize(value):
    """
    Keep the numbers as large as possible while preventing
    long values from overflowing the heatmap cells.
    """
    digits = len(str(abs(int(value))))

    if digits <= 2:
        return COUNT_FONTSIZE

    if digits == 3:
        return COUNT_FONTSIZE - 3

    return COUNT_FONTSIZE - 5


def draw_legend(ax):
    x0 = 1.04

    box_width = 0.050
    box_height = 0.030
    box_gap = 0.009

    legend_rows = [
        ('+', 'increase', 0.93),
        ('−', 'decrease', 0.85),
        ('T', 'tie', 0.77),
    ]

    for symbol, direction, y in legend_rows:
        ax.text(
            x0,
            y + box_height / 2,
            symbol,
            transform=ax.transAxes,
            fontsize=16,
            fontweight='bold',
            va='center',
            clip_on=False,
        )

        cmap = {
            'increase': colormaps['Reds'],
            'decrease': colormaps['Blues'],
            'tie': colormaps['Greys'],
        }[direction]

        for k in range(5):
            frac = (k + 1) / 5

            ax.add_patch(
                Rectangle(
                    (
                        x0
                        + 0.06
                        + k * (box_width + box_gap),
                        y,
                    ),
                    box_width,
                    box_height,
                    transform=ax.transAxes,
                    facecolor=cmap(
                        0.18 + 0.82 * frac
                    ),
                    edgecolor='#BFBFBF',
                    linewidth=0.4,
                    clip_on=False,
                )
            )

    legend_left = x0 + 0.06

    legend_right = (
        legend_left
        + 4 * (box_width + box_gap)
        + box_width
    )

    ax.text(
        legend_left,
        0.715,
        'low',
        transform=ax.transAxes,
        fontsize=LEGEND_FONTSIZE,
        ha='left',
        va='top',
        clip_on=False,
    )

    ax.text(
        legend_right,
        0.715,
        'high',
        transform=ax.transAxes,
        fontsize=LEGEND_FONTSIZE,
        ha='right',
        va='top',
        clip_on=False,
    )


def draw_panel(
    ax,
    counts,
    loci,
    title,
    panel_label,
    add_legend=False,
):
    loci = cluster_loci(
        counts,
        loci,
    )

    values, fracs = matrix_for_loci(
        counts,
        loci,
    )

    n = len(loci)

    # Draw heatmap cells.
    for i, locus in enumerate(loci):
        y = n - 1 - i

        for j, x in enumerate(XPOS):
            direction = COLS[j][1]
            frac = fracs[i, j]
            value = values[i, j]

            rect = Rectangle(
                (
                    x - CELL_W / 2,
                    y - CELL_H / 2,
                ),
                CELL_W,
                CELL_H,
                facecolor=get_color(
                    direction,
                    frac,
                ),
                edgecolor='#D0D0D0',
                linewidth=0.9,
            )

            ax.add_patch(rect)

            if value > 0:
                text_color = (
                    'white'
                    if frac >= 0.58
                    else 'black'
                )

                ax.text(
                    x,
                    y,
                    str(value),
                    ha='center',
                    va='center',
                    fontsize=get_number_fontsize(value),
                    fontweight='bold',
                    color=text_color,
                )

    # Axes and labels.
    ax.set_xlim(-0.9, 6.9)
    ax.set_ylim(-0.7, n - 0.3)

    ax.set_xticks(XPOS)

    ax.set_xticklabels(
        ['+', '−', 'T', '+', '−', 'T'],
        fontsize=XTICK_FONTSIZE,
        fontweight='bold',
    )

    ax.set_yticks(
        range(n)
    )

    ax.set_yticklabels(
        [
            DISPLAY.get(locus, locus)
            for locus in loci[::-1]
        ],
        fontsize=GENE_FONTSIZE,
    )

    ax.tick_params(
        axis='both',
        which='both',
        length=0,
        pad=7,
    )

    ax.set_title(
        title,
        fontsize=TITLE_FONTSIZE,
        fontweight='bold',
        pad=23,
    )

    # Assay headers.
    ax.text(
        1.0,
        1.025,
        'THP-1',
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='bottom',
        fontsize=GROUP_FONTSIZE,
        fontweight='bold',
    )

    ax.text(
        5.0,
        1.025,
        'Brain',
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='bottom',
        fontsize=GROUP_FONTSIZE,
        fontweight='bold',
    )

    # Separation between THP-1 and Brain.
    ax.axvspan(
        2.48,
        3.52,
        color='white',
        zorder=-5,
    )

    ax.axvline(
        3.0,
        color='#AFAFAF',
        linewidth=1.2,
    )

    # Panel border.
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_edgecolor('#909090')

    # Panel label.
    ax.text(
        -0.20,
        1.055,
        panel_label,
        transform=ax.transAxes,
        fontsize=PANEL_FONTSIZE,
        fontweight='bold',
        va='bottom',
        ha='left',
    )

    if add_legend:
        draw_legend(ax)


def main():
    set_style()

    DATA.mkdir(
        parents=True,
        exist_ok=True,
    )

    counts = load_counts()

    fig = plt.figure(
        figsize=(17.0, 9.4)
    )

    grid = fig.add_gridspec(
        1,
        2,
        width_ratios=[1, 1],
        wspace=0.43,
        left=0.13,
        right=0.89,
        top=0.89,
        bottom=0.09,
    )

    ax1 = fig.add_subplot(
        grid[0, 0]
    )

    ax2 = fig.add_subplot(
        grid[0, 1]
    )

    draw_panel(
        ax1,
        counts,
        GWAS_LOCI,
        'GWAS loci',
        'a',
        add_legend=False,
    )

    draw_panel(
        ax2,
        counts,
        WGS_LOCI,
        'WGS loci',
        'b',
        add_legend=True,
    )

    fig.savefig(
        OUT_PNG,
        dpi=600,
        bbox_inches='tight',
        facecolor='white',
    )

    fig.savefig(
        OUT_PDF,
        bbox_inches='tight',
        facecolor='white',
    )

    plt.close(fig)

    print('Saved to:', OUT_PNG)
    print('Saved to:', OUT_PDF)


if __name__ == '__main__':
    main()

Using font: Arial
Saved to: risk_allele_plot/ad_common_augmented_main_heatmap_with_tie_direction_corrected.png
Saved to: risk_allele_plot/ad_common_augmented_main_heatmap_with_tie_direction_corrected.pdf


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colormaps, font_manager
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage


# Put this script and the input CSV files in the same directory, or edit the paths below.
DATA = Path(__file__).resolve().parent


def first_existing(*candidates: str) -> Path:
    """Return the first existing candidate path, relative to DATA when needed."""
    checked = []
    for candidate in candidates:
        path = Path(candidate)
        if not path.is_absolute():
            path = DATA / path
        checked.append(str(path))
        if path.exists():
            return path
    raise FileNotFoundError('None of these files exists:\n' + '\n'.join(checked))


THP1_FILE = first_existing(
    'annotated_20240616_comparative_THP1Macrophage_alleleOnly(3).csv',
    'annotated_20240616_comparative_THP1Macrophage_alleleOnly.csv',
    'allele_differences_withoutcontrol/for_plotting_allele_only_20240617/'
    'annotated_20240616_comparative_THP1Macrophage_alleleOnly.csv',
)
BRAIN_FILE = first_existing(
    'annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly(3).csv',
    'annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly.csv',
    'allele_differences_withoutcontrol/for_plotting_allele_only_20240617/'
    'annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly.csv',
)
LOCUS_MAP_FILE = first_existing(
    'variant_level_emvar_calls_gene_annotation.csv',
    'risk_allele_plot/variant_level_emvar_calls_gene_annotation.csv',
)
CLASS_FILE = first_existing(
    'gwas_wgs_locus_source_classification.csv',
    'risk_allele_plot/gwas_wgs_locus_source_classification.csv',
)
MODEL_FILE = first_existing(
    'SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_'
    'Bellenguez_TSS_genomicML_logFC_diff_MAFCorrect_REFindex_catlas_'
    'with_all_peakfiles_variant_category_20260310_Bellenguez_AD_direction(6).csv',
)

OUT_PNG = DATA / 'minor_allele_mpra_main_ml_parentheses_mpra_color_direction_corrected.png'
OUT_PDF = DATA / 'minor_allele_mpra_main_ml_parentheses_mpra_color_direction_corrected.pdf'
OUT_VARIANT_VOTES = DATA / 'minor_allele_variant_votes_mpra_main_ml_parentheses_mpra_color_direction_corrected.csv'
OUT_LOCUS_COUNTS = DATA / 'minor_allele_locus_counts_mpra_main_ml_parentheses_mpra_color_direction_corrected.csv'
OUT_LOCUS_TOTALS = DATA / 'minor_allele_locus_tested_totals_mpra_main_ml_parentheses_mpra_color_direction_corrected.csv'
OUT_MODEL_SUPPORT = DATA / 'variant_model_support_flags_mpra_main_ml_parentheses_mpra_color_direction_corrected.csv'
OUT_DIAGNOSTICS = DATA / 'model_support_merge_diagnostics_mpra_main_ml_parentheses_mpra_color_direction_corrected.csv'


# Model columns: any non-null diff_genomic value counts as a significant prediction.
THP1_MODEL_COLUMNS = [
    'ML_majorminor_diff_genomic_THP1_IFNB-Calvin',
    'ML_majorminor_diff_genomic_THP1_IFNG-Calvin',
    'ML_majorminor_diff_genomic_THP1_LPSIFNG-Calvin',
    'ML_majorminor_diff_genomic_THP1_Naive-Calvin',
]
BRAIN_MODEL_COLUMNS = [
    'ML_majorminor_diff_genomic_fullard_DLPFC-model-best',
    'ML_majorminor_diff_genomic_fullard_putamen-model-best',
    'ML_majorminor_diff_genomic_fullard_hippocampus-model-best',
]


# Curated loci displayed in the main figure.
GWAS_LOCI = [
    'HLA-DRB1',
    'BIN1',
    'MS4A4A',
    'ABCA7',
    'CLU',
    'SCIMP',
    'HLA-DQA1',
    'SORL1',
    'PTK2B',
    'APOC2',
]

WGS_LOCI = [
    'APOE',
    'SEC63',
    'NELL2',
    'DBX2',
    'POU3F3',
    'IL1F10',
    'ST8SIA4',
    'SNAP25',
    'BTBD2',
    'PRKCH',
]

DISPLAY = {
    'SEC63': 'SEC63–OSTM1',
}

ASSAYS = [
    'THP-1 macrophages',
    'Brain tissue',
]

COLS = [
    ('THP-1 macrophages', 'increase'),
    ('THP-1 macrophages', 'decrease'),
    ('THP-1 macrophages', 'tie'),
    ('Brain tissue', 'increase'),
    ('Brain tissue', 'decrease'),
    ('Brain tissue', 'tie'),
]

XPOS = [0.0, 1.0, 2.0, 4.0, 5.0, 6.0]
FDR_THRESHOLD = 0.05

# Figure typography.
GENE_FONTSIZE = 14
MAIN_COUNT_FONTSIZE = 16
ML_COUNT_FONTSIZE = 9
XTICK_FONTSIZE = 15
GROUP_FONTSIZE = 17
TITLE_FONTSIZE = 20
PANEL_FONTSIZE = 24
LEGEND_FONTSIZE = 10
CELL_W = 0.96
CELL_H = 0.94


def set_style():
    preferred_fonts = ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans']
    available = {font.name for font in font_manager.fontManager.ttflist}
    chosen = next((font for font in preferred_fonts if font in available), 'DejaVu Sans')

    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': [chosen] + preferred_fonts,
        'font.size': 15,
        'axes.titlesize': TITLE_FONTSIZE,
        'xtick.labelsize': XTICK_FONTSIZE,
        'ytick.labelsize': GENE_FONTSIZE,
        'axes.linewidth': 1.0,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'svg.fonttype': 'none',
        'savefig.dpi': 600,
    })
    print('Using font:', chosen)


def read_assay(path: Path, assay: str) -> pd.DataFrame:
    required = [
        'Unnamed: 0', 'rsid', 'center', 'Major', 'Minor', 'fdr', 'logFC_converted'
    ]
    df = pd.read_csv(path, usecols=required)
    df = df.rename(columns={'Unnamed: 0': 'construct_id'})
    df['assay'] = assay
    for column in ['rsid', 'Major', 'Minor']:
        df[column] = df[column].astype(str)
    df['fdr'] = pd.to_numeric(df['fdr'], errors='coerce')
    df['logFC_converted'] = pd.to_numeric(df['logFC_converted'], errors='coerce')
    df = df[df['center'].isin(['SNPCENTER', 'PEAKCENTER'])].copy()
    df = (
        df.sort_values('fdr')
        .drop_duplicates(
            ['assay', 'construct_id', 'rsid', 'Major', 'Minor'],
            keep='first',
        )
    )
    return df


def attach_locus(all_constructs: pd.DataFrame) -> pd.DataFrame:
    locus_map = pd.read_csv(LOCUS_MAP_FILE, usecols=['rsID', 'locus'])
    locus_map['rsID'] = locus_map['rsID'].astype(str)
    locus_map = locus_map.drop_duplicates('rsID')
    merged = all_constructs.merge(
        locus_map,
        left_on='rsid',
        right_on='rsID',
        how='left',
        validate='many_to_one',
    )
    if merged['locus'].isna().any():
        missing = merged.loc[merged['locus'].isna(), 'rsid'].drop_duplicates().tolist()
        print(
            f'Dropping {len(missing)} IDs without a final locus annotation. '
            f'Examples: {", ".join(missing[:10])}'
        )
        merged = merged[merged['locus'].notna()].copy()
    return merged


def calculate_tested_totals(all_constructs: pd.DataFrame) -> pd.DataFrame:
    totals = (
        all_constructs[['locus', 'rsid', 'Major', 'Minor']]
        .drop_duplicates()
        .groupby('locus', as_index=False)
        .size()
        .rename(columns={'size': 'n_tested_comparisons'})
    )
    totals.to_csv(OUT_LOCUS_TOTALS, index=False)
    return totals


def load_model_support() -> pd.DataFrame:
    required = ['rsid', 'Major', 'Minor'] + THP1_MODEL_COLUMNS + BRAIN_MODEL_COLUMNS
    model = pd.read_csv(MODEL_FILE, usecols=required)
    for column in ['rsid', 'Major', 'Minor']:
        model[column] = model[column].astype(str)

    missing_columns = [c for c in THP1_MODEL_COLUMNS + BRAIN_MODEL_COLUMNS if c not in model.columns]
    if missing_columns:
        raise ValueError('Missing model columns: ' + ', '.join(missing_columns))

    # Any non-null diff_genomic value is treated as a significant prediction,
    # irrespective of effect direction.
    model['thp1_model_supported'] = model[THP1_MODEL_COLUMNS].notna().any(axis=1)
    model['brain_model_supported'] = model[BRAIN_MODEL_COLUMNS].notna().any(axis=1)

    # Multiple rows may correspond to the same allele comparison. Collapse with ANY.
    support = (
        model.groupby(['rsid', 'Major', 'Minor'], as_index=False)[
            ['thp1_model_supported', 'brain_model_supported']
        ]
        .any()
    )
    support.to_csv(OUT_MODEL_SUPPORT, index=False)
    return support


def vote_minor_allele_direction(all_constructs: pd.DataFrame) -> pd.DataFrame:
    """
    Assign each significant construct to the direction of the MINOR allele.

    The input definition is:
        logFC_converted = Major allele activity - Minor allele activity

    Therefore:
        logFC_converted < 0  -> Minor > Major -> minor allele increases transcription
        logFC_converted > 0  -> Major > Minor -> minor allele decreases transcription

    Construct-level directions are then combined by majority vote for each
    assay/locus/rsid/Major/Minor allele comparison. Equal positive and negative
    construct counts are classified as a tie.
    """
    significant = all_constructs[
        all_constructs['fdr'].lt(FDR_THRESHOLD)
        & all_constructs['logFC_converted'].notna()
        & all_constructs['logFC_converted'].ne(0)
    ].copy()

    significant['construct_direction'] = np.where(
        significant['logFC_converted'] < 0,
        'increase',
        'decrease',
    )

    records = []
    group_cols = ['assay', 'locus', 'rsid', 'Major', 'Minor']
    for keys, group in significant.groupby(group_cols, dropna=False):
        assay, locus, rsid, major, minor = keys
        n_increase = int((group['construct_direction'] == 'increase').sum())
        n_decrease = int((group['construct_direction'] == 'decrease').sum())

        if n_increase > n_decrease:
            vote = 'increase'
        elif n_decrease > n_increase:
            vote = 'decrease'
        else:
            vote = 'tie'

        records.append({
            'assay': assay,
            'locus': locus,
            'rsid': rsid,
            'Major': major,
            'Minor': minor,
            'n_significant_constructs': len(group),
            'n_increase_constructs': n_increase,
            'n_decrease_constructs': n_decrease,
            'vote': vote,
            'construct_ids': ';'.join(sorted(group['construct_id'].astype(str))),
        })

    return pd.DataFrame(records)


def attach_model_support(votes: pd.DataFrame, model_support: pd.DataFrame) -> pd.DataFrame:
    merged = votes.merge(
        model_support,
        on=['rsid', 'Major', 'Minor'],
        how='left',
        validate='many_to_one',
        indicator=True,
    )
    merged['model_row_matched'] = merged['_merge'].eq('both')
    merged = merged.drop(columns='_merge')
    merged[['thp1_model_supported', 'brain_model_supported']] = (
        merged[['thp1_model_supported', 'brain_model_supported']]
        .fillna(False)
        .astype(bool)
    )
    merged['model_supported'] = np.where(
        merged['assay'].eq('THP-1 macrophages'),
        merged['thp1_model_supported'],
        merged['brain_model_supported'],
    ).astype(bool)

    diagnostics = (
        merged.groupby('assay', as_index=False)
        .agg(
            mpra_emvar_comparisons=('rsid', 'size'),
            exact_model_rows_matched=('model_row_matched', 'sum'),
            model_supported=('model_supported', 'sum'),
        )
    )
    diagnostics['exact_match_fraction'] = (
        diagnostics['exact_model_rows_matched'] / diagnostics['mpra_emvar_comparisons']
    )
    diagnostics.to_csv(OUT_DIAGNOSTICS, index=False)
    merged.to_csv(OUT_VARIANT_VOTES, index=False)
    return merged


def load_counts():
    thp1 = read_assay(THP1_FILE, 'THP-1 macrophages')
    brain = read_assay(BRAIN_FILE, 'Brain tissue')
    all_constructs = pd.concat([thp1, brain], ignore_index=True)
    all_constructs = attach_locus(all_constructs)

    totals = calculate_tested_totals(all_constructs)
    model_support = load_model_support()

    locus_classes = pd.read_csv(CLASS_FILE, usecols=['locus', 'source_class'])
    locus_classes = locus_classes.drop_duplicates('locus')

    votes = vote_minor_allele_direction(all_constructs)
    votes = attach_model_support(votes, model_support)
    votes = votes.merge(
        locus_classes,
        on='locus',
        how='left',
        validate='many_to_one',
    )
    votes = votes[votes['source_class'].isin(['GWAS', 'WGS'])].copy()

    # Count all MPRA emVars and the model-supported subset separately.
    grouped = (
        votes.groupby(['locus', 'source_class', 'assay', 'vote'], as_index=False)
        .agg(
            mpra_count=('rsid', 'size'),
            model_supported_count=('model_supported', 'sum'),
        )
    )
    wide = grouped.pivot_table(
        index=['locus', 'source_class', 'assay'],
        columns='vote',
        values=['mpra_count', 'model_supported_count'],
        fill_value=0,
        aggfunc='sum',
    )
    wide.columns = [f'{metric}_{direction}' for metric, direction in wide.columns]
    wide = wide.reset_index()

    rows = []
    for locus in GWAS_LOCI:
        for assay in ASSAYS:
            rows.append((locus, 'GWAS', assay))
    for locus in WGS_LOCI:
        for assay in ASSAYS:
            rows.append((locus, 'WGS', assay))
    base = pd.DataFrame(rows, columns=['locus', 'source_class', 'assay'])
    counts = base.merge(wide, on=['locus', 'source_class', 'assay'], how='left')

    for direction in ['increase', 'decrease', 'tie']:
        for prefix in ['mpra_count', 'model_supported_count']:
            column = f'{prefix}_{direction}'
            if column not in counts.columns:
                counts[column] = 0
            counts[column] = counts[column].fillna(0).astype(int)

    # Color intensity is determined only by the MPRA distribution within each
    # locus and assay, matching the original heatmap logic. The denominator is
    # increase + decrease + tie MPRA emVars for that locus-assay combination.
    counts['mpra_total_in_assay'] = (
        counts['mpra_count_increase']
        + counts['mpra_count_decrease']
        + counts['mpra_count_tie']
    )
    for direction in ['increase', 'decrease', 'tie']:
        counts[f'mpra_fraction_{direction}'] = np.where(
            counts['mpra_total_in_assay'] > 0,
            counts[f'mpra_count_{direction}'] / counts['mpra_total_in_assay'],
            0.0,
        )

    counts = counts.merge(totals, on='locus', how='left')
    counts['n_tested_comparisons'] = counts['n_tested_comparisons'].fillna(0).astype(int)
    counts.to_csv(OUT_LOCUS_COUNTS, index=False)
    return counts


def matrix_for_loci(counts: pd.DataFrame, loci: list[str]):
    lookup = counts.set_index(['locus', 'assay'])
    model_values = np.zeros((len(loci), len(COLS)), dtype=int)
    mpra_values = np.zeros((len(loci), len(COLS)), dtype=int)
    mpra_fracs = np.zeros((len(loci), len(COLS)), dtype=float)

    for i, locus in enumerate(loci):
        for j, (assay, direction) in enumerate(COLS):
            row = lookup.loc[(locus, assay)]
            model_values[i, j] = int(row[f'model_supported_count_{direction}'])
            mpra_values[i, j] = int(row[f'mpra_count_{direction}'])
            mpra_fracs[i, j] = float(row[f'mpra_fraction_{direction}'])

    return model_values, mpra_values, mpra_fracs


def cluster_loci(counts: pd.DataFrame, loci: list[str]):
    if len(loci) <= 2:
        return loci
    model_values, mpra_values, mpra_fracs = matrix_for_loci(counts, loci)
    mpra_totals = mpra_values.sum(axis=1, keepdims=True)
    mpra_totals = np.where(mpra_totals == 0, 1, mpra_totals)
    model_totals = model_values.sum(axis=1, keepdims=True)
    model_totals = np.where(model_totals == 0, 1, model_totals)
    features = np.concatenate(
        [mpra_fracs, mpra_values / mpra_totals, model_values / model_totals],
        axis=1,
    )
    try:
        linkage_matrix = linkage(
            features,
            method='average',
            metric='euclidean',
            optimal_ordering=True,
        )
        return [loci[i] for i in leaves_list(linkage_matrix)]
    except Exception as error:
        print('Clustering failed:', error)
        return loci


def get_color(direction: str, fraction: float):
    if fraction <= 0:
        return (1, 1, 1, 1)
    cmap = {
        'increase': colormaps['Reds'],
        'decrease': colormaps['Blues'],
        'tie': colormaps['Greys'],
    }[direction]
    return cmap(0.18 + 0.82 * fraction)


def draw_cell_numbers(ax, x, y, model_count, mpra_count, mpra_fraction):
    """Main number = total MPRA emVars; parentheses = ML-supported subset."""
    if mpra_count <= 0:
        return
    text_color = 'white' if mpra_fraction >= 0.62 else 'black'
    ax.text(
        x,
        y + 0.10,
        str(mpra_count),
        ha='center',
        va='center',
        fontsize=MAIN_COUNT_FONTSIZE,
        fontweight='bold',
        color=text_color,
    )
    ax.text(
        x,
        y - 0.20,
        f'({model_count})',
        ha='center',
        va='center',
        fontsize=ML_COUNT_FONTSIZE,
        color=text_color,
    )


def draw_legend(ax):
    x0 = 1.04
    box_width = 0.050
    box_height = 0.030
    box_gap = 0.009
    rows = [
        ('+', 'increase', 0.93),
        ('−', 'decrease', 0.85),
        ('T', 'tie', 0.77),
    ]
    for symbol, direction, y in rows:
        ax.text(
            x0, y + box_height / 2, symbol,
            transform=ax.transAxes, fontsize=15, fontweight='bold',
            va='center', clip_on=False,
        )
        cmap = {
            'increase': colormaps['Reds'],
            'decrease': colormaps['Blues'],
            'tie': colormaps['Greys'],
        }[direction]
        for k in range(5):
            fraction = (k + 1) / 5
            ax.add_patch(
                Rectangle(
                    (x0 + 0.055 + k * (box_width + box_gap), y),
                    box_width,
                    box_height,
                    transform=ax.transAxes,
                    facecolor=cmap(0.18 + 0.82 * fraction),
                    edgecolor='#BFBFBF',
                    linewidth=0.35,
                    clip_on=False,
                )
            )
    left = x0 + 0.055
    right = left + 4 * (box_width + box_gap) + box_width
    ax.text(left, 0.72, 'low', transform=ax.transAxes, fontsize=LEGEND_FONTSIZE,
            ha='left', va='top', clip_on=False)
    ax.text(right, 0.72, 'high', transform=ax.transAxes, fontsize=LEGEND_FONTSIZE,
            ha='right', va='top', clip_on=False)
    ax.text(
        x0,
        0.66,
        'MPRA total\n(ML-supported)',
        transform=ax.transAxes,
        fontsize=LEGEND_FONTSIZE,
        ha='left',
        va='top',
        clip_on=False,
    )
    ax.text(
        x0,
        0.56,
        'Color: MPRA fraction',
        transform=ax.transAxes,
        fontsize=LEGEND_FONTSIZE,
        ha='left',
        va='top',
        clip_on=False,
    )


def draw_panel(ax, counts, loci, title, panel_label, add_legend=False):
    loci = cluster_loci(counts, loci)
    model_values, mpra_values, mpra_fracs = matrix_for_loci(counts, loci)
    n = len(loci)

    tested_lookup = (
        counts[['locus', 'n_tested_comparisons']]
        .drop_duplicates('locus')
        .set_index('locus')['n_tested_comparisons']
    )

    for i, locus in enumerate(loci):
        y = n - 1 - i
        for j, x in enumerate(XPOS):
            direction = COLS[j][1]
            fraction = mpra_fracs[i, j]
            model_count = model_values[i, j]
            mpra_count = mpra_values[i, j]

            ax.add_patch(
                Rectangle(
                    (x - CELL_W / 2, y - CELL_H / 2),
                    CELL_W,
                    CELL_H,
                    facecolor=get_color(direction, fraction),
                    edgecolor='#D0D0D0',
                    linewidth=0.9,
                )
            )
            draw_cell_numbers(ax, x, y, model_count, mpra_count, fraction)

    ax.set_xlim(-0.9, 6.9)
    ax.set_ylim(-0.7, n - 0.3)
    ax.set_xticks(XPOS)
    ax.set_xticklabels(['+', '−', 'T', '+', '−', 'T'],
                       fontsize=XTICK_FONTSIZE, fontweight='bold')
    ax.set_yticks(range(n))
    ax.set_yticklabels(
        [
            f"{DISPLAY.get(locus, locus)} ({int(tested_lookup.loc[locus])})"
            for locus in loci[::-1]
        ],
        fontsize=GENE_FONTSIZE,
    )
    ax.tick_params(axis='both', which='both', length=0, pad=7)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold', pad=23)

    ax.text(
        1.0, 1.025, 'THP-1', transform=ax.get_xaxis_transform(),
        ha='center', va='bottom', fontsize=GROUP_FONTSIZE, fontweight='bold',
    )
    ax.text(
        5.0, 1.025, 'Brain', transform=ax.get_xaxis_transform(),
        ha='center', va='bottom', fontsize=GROUP_FONTSIZE, fontweight='bold',
    )

    ax.axvspan(2.48, 3.52, color='white', zorder=-5)
    ax.axvline(3.0, color='#AFAFAF', linewidth=1.2)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_edgecolor('#909090')

    ax.text(
        -0.18, 1.055, panel_label, transform=ax.transAxes,
        fontsize=PANEL_FONTSIZE, fontweight='bold', va='bottom', ha='left',
    )

    if add_legend:
        draw_legend(ax)


def main():
    set_style()
    counts = load_counts()

    fig = plt.figure(figsize=(17.0, 9.4))
    grid = fig.add_gridspec(
        1, 2,
        width_ratios=[1, 1],
        wspace=0.43,
        left=0.13,
        right=0.89,
        top=0.89,
        bottom=0.09,
    )

    ax1 = fig.add_subplot(grid[0, 0])
    ax2 = fig.add_subplot(grid[0, 1])
    draw_panel(ax1, counts, GWAS_LOCI, 'GWAS loci', 'a', add_legend=False)
    draw_panel(ax2, counts, WGS_LOCI, 'WGS loci', 'b', add_legend=True)

    fig.savefig(OUT_PNG, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(OUT_PDF, bbox_inches='tight', facecolor='white')
    plt.close(fig)

    print('Saved to:', OUT_PNG)
    print('Saved to:', OUT_PDF)
    print('Saved variant votes to:', OUT_VARIANT_VOTES)
    print('Saved locus counts to:', OUT_LOCUS_COUNTS)
    print('Saved model support flags to:', OUT_MODEL_SUPPORT)
    print('Saved diagnostics to:', OUT_DIAGNOSTICS)


if __name__ == '__main__':
    main()